In [1]:
import os
import json
from PIL import Image
import matplotlib.pyplot as plt
import torch
from torch import nn, optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms, models, ops
from torchvision.models import resnet18, ResNet18_Weights
from torchinfo import summary
from scipy.io import loadmat
import numpy as np
import tqdm
import math
import torch.backends.cudnn as cudnn

class AverageMeter(object):
    """Computes and stores the average and current value"""
    def __init__(self, name=''):
        self.name = name
        self.reset()
    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0
    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count

def load_tango_3d_keypoints(mat_dir):
    vertices  = loadmat(mat_dir)['tango3Dpoints'] # [3 x 11]
    corners3D = np.transpose(np.array(vertices, dtype=np.float32)) # [11 x 3]
    return corners3D

def load_camera_intrinsics(camera_json):
    with open(camera_json) as f:
        cam = json.load(f)
    cameraMatrix = np.array(cam['cameraMatrix'], dtype=np.float32)
    distCoeffs   = np.array(cam['distCoeffs'],   dtype=np.float32)
    return cameraMatrix, distCoeffs

def quat2dcm(q):
    """ Computing direction cosine matrix from quaternion. """
    q = q/np.linalg.norm(q)
    q0, q1, q2, q3 = q[0], q[1], q[2], q[3]
    dcm = np.zeros((3, 3))
    dcm[0, 0] = 2 * q0 ** 2 - 1 + 2 * q1 ** 2
    dcm[1, 1] = 2 * q0 ** 2 - 1 + 2 * q2 ** 2
    dcm[2, 2] = 2 * q0 ** 2 - 1 + 2 * q3 ** 2
    dcm[0, 1] = 2 * q1 * q2 + 2 * q0 * q3
    dcm[0, 2] = 2 * q1 * q3 - 2 * q0 * q2
    dcm[1, 0] = 2 * q1 * q2 - 2 * q0 * q3
    dcm[1, 2] = 2 * q2 * q3 + 2 * q0 * q1
    dcm[2, 0] = 2 * q1 * q3 + 2 * q0 * q2
    dcm[2, 1] = 2 * q2 * q3 - 2 * q0 * q1
    return dcm

def project_keypoints(q_vbs2tango, r_Vo2To_vbs, cameraMatrix, distCoeffs, keypoints):
    if keypoints.shape[0] != 3:
        keypoints = np.transpose(keypoints)
    
    keypoints = np.vstack((keypoints, np.ones((1, keypoints.shape[1]))))
    
    pose_mat = np.hstack((np.transpose(quat2dcm(q_vbs2tango)),
                          np.expand_dims(r_Vo2To_vbs, 1)))
    xyz      = np.dot(pose_mat, keypoints)
    x0, y0   = xyz[0,:] / xyz[2,:], xyz[1,:] / xyz[2,:]

    r2 = x0*x0 + y0*y0
    cdist = 1 + distCoeffs[0]*r2 + distCoeffs[1]*r2*r2 + distCoeffs[4]*r2*r2*r2
    x  = x0*cdist + distCoeffs[2]*2*x0*y0 + distCoeffs[3]*(r2 + 2*x0*x0)
    y  = y0*cdist + distCoeffs[2]*(r2 + 2*y0*y0) + distCoeffs[3]*2*x0*y0

    points2D = np.vstack((cameraMatrix[0,0]*x + cameraMatrix[0,2],
                          cameraMatrix[1,1]*y + cameraMatrix[1,2]))
    return points2D

# --- Dataset ---

class Speed(Dataset):
    def __init__(self, images_dir, json_dir, mat_dir='/kaggle/input/mat-file/tangoPoints.mat', camera_json='/kaggle/input/mat-file/camera.json', transform=None):
        self.images_dir = images_dir
        self.json_dir = json_dir
        self.transform = transform
        self.imagesList = []
        self.BBoxList = []
        self.keypointsList = []

        # Placeholder loading logic - assume files exist or mocks will be needed if running without data
        if not os.path.exists(json_dir):
            print(f"Warning: JSON file {json_dir} not found. Creating empty dataset for syntax check.")
            return

        # Load intrinsics and 3D points
        try:
             keypts3d = load_tango_3d_keypoints(mat_dir)
             cameraMatrix, distCoeffs = load_camera_intrinsics(camera_json)
        except:
             # Basic mock if files missing
             keypts3d = np.zeros((11, 3), dtype=np.float32)
             cameraMatrix, distCoeffs = np.eye(3, dtype=np.float32), np.zeros(5, dtype=np.float32)

        with open(self.json_dir, 'r') as f:
            annotations = json.load(f)
            lookup = { item['filename']: item for item in annotations }

            cnt = 0
            # Assuming os.listdir might fail if dir doesn't exist, wrap or assume exist
            if os.path.exists(self.images_dir):
                file_list = os.listdir(self.images_dir)
            else:
                file_list = []

            for filename in tqdm.tqdm(file_list):
                if filename not in lookup: continue
                self.imagesList.append(os.path.join(self.images_dir, filename))
                
                q = np.array(lookup[filename]["q_vbs2tango"], dtype=np.float32)
                r = np.array(lookup[filename]['r_Vo2To_vbs_true'], dtype=np.float32)
                
                keypts2d = project_keypoints(q, r, cameraMatrix, distCoeffs, keypts3d)
                self.keypointsList.append(torch.tensor(keypts2d, dtype=torch.float32))
                
                xmin, xmax = np.min(keypts2d[0]), np.max(keypts2d[0])
                ymin, ymax = np.min(keypts2d[1]), np.max(keypts2d[1])
                
                w, h = xmax - xmin, ymax - ymin
                margin_x, margin_y = w * 0.05, h * 0.05
                
                xmin, xmax = max(0, xmin - margin_x), xmax + margin_x
                ymin, ymax = max(0, ymin - margin_y), ymax + margin_y
                
                self.BBoxList.append(torch.tensor([xmin, xmax, ymin, ymax], dtype=torch.float32))
                
                cnt += 1
                # if cnt > 400: break 

    def __len__(self):
        return len(self.imagesList)

    def __getitem__(self, idx):
        image_path = self.imagesList[idx]
        image = Image.open(image_path).convert('RGB')
        orig_w, orig_h = image.size
        BBox = self.BBoxList[idx] # xmin, xmax, ymin, ymax
        
        # Resize to fixed size
        target_size = (224, 224)
        image = image.resize(target_size)
        scale_x = target_size[0] / orig_w
        scale_y = target_size[1] / orig_h
        
        xmin, xmax, ymin, ymax = BBox.tolist()
        xmin *= scale_x
        xmax *= scale_x
        ymin *= scale_y
        ymax *= scale_y
        
        # Convert to center format [cx, cy, w, h] normalized
        w = xmax - xmin
        h = ymax - ymin
        cx = xmin + w / 2
        cy = ymin + h / 2
        
        # Normalize by image size
        box_norm = torch.tensor([cx / target_size[0], cy / target_size[1], 
                                 w / target_size[0], h / target_size[1]], dtype=torch.float32)
        
        if self.transform:
            image = self.transform(image)
            
        return image, box_norm


class LW_Detr(nn.Module):
    def __init__(self, d_model=256, nhead=8, num_layers=3):
        super().__init__()
        # Backbone: ResNet18
        resnet = resnet18(weights=ResNet18_Weights.DEFAULT)
        self.backbone = nn.Sequential(*list(resnet.children())[:-2]) # Remove AvgPool and FC
        self.conv = nn.Conv2d(512, d_model, 1) # Project 512 channels to d_model

        # Transformer
        self.transformer = nn.Transformer(
            d_model=d_model, 
            nhead=nhead, 
            num_encoder_layers=num_layers, 
            num_decoder_layers=num_layers,
            batch_first=True
        )
        
        # Embeddings
        self.query_pos = nn.Parameter(torch.rand(1, d_model)) # One object query
        self.row_embed = nn.Parameter(torch.rand(50, d_model // 2))
        self.col_embed = nn.Parameter(torch.rand(50, d_model // 2))
        
        # Prediction Heads
        self.bbox_head = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.ReLU(),
            nn.Linear(d_model, 4),
            nn.Sigmoid() # Output 0-1
        )
        
    def forward(self, x):
        # x: [B, 3, H, W]
        features = self.backbone(x) # [B, 512, H/32, W/32]
        h = self.conv(features) # [B, d_model, H', W']
        B, C, H, W = h.shape
        
        # Flatten
        src = h.flatten(2).permute(0, 2, 1) # [B, H*W, C]
        
        # Positional Encoding
        pos_embed = torch.cat([
            self.col_embed[:W].unsqueeze(0).repeat(H, 1, 1),
            self.row_embed[:H].unsqueeze(1).repeat(1, W, 1),
        ], dim=-1).flatten(0, 1).unsqueeze(0).repeat(B, 1, 1) # [B, H*W, C]
        
        query_embed = self.query_pos.unsqueeze(0).repeat(B, 1, 1) # [B, 1, C]
        
        # Transformer
        out = self.transformer(src + pos_embed, query_embed) # [B, 1, C]
        
        # Head
        box_pred = self.bbox_head(out).squeeze(1) # [B, 4]
        return box_pred


def test_loop(test_dataloader, model, device):
    err_iou_meter = AverageMeter('iou')
    iou_errors_all = []
    model.eval()
    
    with torch.no_grad():
        for batch, (X, yBbox) in enumerate(test_dataloader):
            X, yBbox = X.to(device), yBbox.to(device)
            pred_box = model(X) # [B, 4] in cxcywh
            
            # Convert both to xyxy for IoU calculation
            pred_xyxy = ops.box_convert(pred_box, 'cxcywh', 'xyxy')
            gt_xyxy = ops.box_convert(yBbox, 'cxcywh', 'xyxy')
            
            # Calculate IoU
            # ops.box_iou returns [N, M] matrix. We want the diagonal for pairs.
            iou_matrix = ops.box_iou(pred_xyxy, gt_xyxy)
            iou = torch.diag(iou_matrix) # [B]
            
            for i in iou.cpu().numpy():
                err_iou_meter.update(i, 1)
                iou_errors_all.append(i)
            print(f"\rBatch {batch+1}/{len(test_dataloader)}: IoU: {err_iou_meter.val:.2f}", end="", flush=True)

    medians = {
        'IoU_med': np.median(iou_errors_all) if iou_errors_all else float('nan')
    }
    return {'IoU': err_iou_meter}, medians


def generalized_box_iou_loss(boxes1, boxes2):
    """
    Generalized IoU Loss.
    boxes1, boxes2: [N, 4] in [cx, cy, w, h] format (normalized 0-1)
    """
    if boxes1.shape[-1] != 4 or boxes2.shape[-1] != 4:
        return F.l1_loss(boxes1, boxes2)

    # Convert to [x1, y1, x2, y2]
    b1_xyxy = ops.box_convert(boxes1, 'cxcywh', 'xyxy')
    b2_xyxy = ops.box_convert(boxes2, 'cxcywh', 'xyxy')
    
    # Calculate GIoU
    giou_matrix = ops.generalized_box_iou(b1_xyxy, b2_xyxy)
    giou = torch.diag(giou_matrix) # Extract diagonal for 1-to-1 matching
    
    return (1 - giou).mean()

def save_checkpoint(states, is_best, output_dir, filename='checkpoint.pth'):
    if not os.path.exists(output_dir): os.makedirs(output_dir)
    torch.save(states, os.path.join(output_dir, filename))
    if is_best and 'state_dict' in states:
        torch.save(states['best_state_dict'], os.path.join(output_dir, 'model_best.pth'))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
cudnn.benchmark = True
    
transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
# Dataset configurations - paths kept as /kaggle for consistency with user environment
images_dir = '/kaggle/input/speedsplit/speed/images/trainval'
val_json = '/kaggle/input/speedsplit/speed/val.json' # Using val.json for testing
real_images_dir = '/kaggle/input/speedsplit/speed/images/real'
real_json = '/kaggle/input/speedsplit/speed/real.json'
mat_file = '/kaggle/input/mat-file/tangoPoints.mat'
camera_json = '/kaggle/input/mat-file/camera.json'

# Load datasets
print("Loading datasets...")
val_dataset = Speed(images_dir, val_json, mat_file, camera_json, transform=transform)
real_dataset = Speed(real_images_dir, real_json, mat_file, camera_json, transform=transform)

batch_size = 16
use_pin_memory = torch.cuda.is_available()
val_dataloader = DataLoader(val_dataset, batch_size, shuffle=False, num_workers=2, pin_memory=use_pin_memory)
real_dataloader = DataLoader(real_dataset, batch_size, shuffle=False, num_workers=2, pin_memory=use_pin_memory)
    
input_dir = '/kaggle/input/fastposedetr'
output_dir = '/kaggle/working/'

model = LW_Detr().to(device)
checkpoint_file = os.path.join(input_dir, 'checkpoint.pth')

if os.path.isfile(checkpoint_file):
    print(f"Loading checkpoint: {checkpoint_file}")
    checkpoint = torch.load(checkpoint_file, weights_only=False, map_location=device) 
    model.load_state_dict(checkpoint['state_dict'])
    print(f"Loaded checkpoint at epoch {checkpoint['epoch']}")
else:
    print(f"Warning: No checkpoint found at {checkpoint_file}. Testing with random weights.")

print("Testing on synthetic dataset...")
performances_syn, medians_syn = test_loop(val_dataloader, model, device)
print("\n")
print("\nTesting on real dataset...")
performances_real, medians_real = test_loop(real_dataloader, model, device)
print("\n")

print("\nResults Summary:")
print("+" + "-"*22 + "+" + "-"*30 + "+" + "-"*30 + "+")
print(f"| {'Metric':<20} | {'SPEED synthetic test-set':<28} | {'SPEED real test-set':<28} |")
print("+" + "-"*22 + "+" + "-"*30 + "+" + "-"*30 + "+")
print(f"| {'Mean IoU (-)':<20} | {performances_syn['IoU'].avg: <28.4f} | {performances_real['IoU'].avg: <28.4f} |")
print(f"| {'Median IoU (-)':<20} | {medians_syn['IoU_med']: <28.4f} | {medians_real['IoU_med']: <28.4f} |")

print("+" + "-"*22 + "+" + "-"*30 + "+" + "-"*30 + "+")

Using device: cpu
Loading datasets...


100%|██████████| 5/5 [00:00<00:00, 1853.26it/s]


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 159MB/s] 


Loading checkpoint: /kaggle/input/fastposedetr/checkpoint.pth
Loaded checkpoint at epoch 70
Testing on synthetic dataset...
Batch 150/150: IoU: 0.98


Testing on real dataset...
Batch 1/1: IoU: 0.94


Results Summary:
+----------------------+------------------------------+------------------------------+
| Metric               | SPEED synthetic test-set     | SPEED real test-set          |
+----------------------+------------------------------+------------------------------+
| Mean IoU (-)         | 0.8817                       | 0.8583                       |
| Median IoU (-)       | 0.9134                       | 0.8219                       |
+----------------------+------------------------------+------------------------------+
